# Notebook 14 — Domain-Based Feature Engineering
**Domain chosen: Telecom** (matches this sprint's dataset). This notebook focuses
purely on **domain thinking** — at least 10 meaningful features, each justified the
way a senior engineer would justify them to a product/business stakeholder, not just
to another engineer.

In [ ]:
import pandas as pd
import numpy as np

customers = pd.read_csv("./telecom_customers.csv", parse_dates=["signup_date"])
transactions = pd.read_csv("./telecom_transactions.csv", parse_dates=["transaction_date"])
REF = pd.Timestamp("2024-06-30")
customers.head(2)

## Domain Context: Why Telecom Churn Is a Distinctive Problem

Telecom subscriptions have a few defining characteristics that should drive feature
design:
- **Contract lock-in** dramatically changes churn risk (Month-to-month vs Two year)
- **Price sensitivity** is high — customers actively compare telecom pricing more than
  most subscription categories
- **Service quality issues** (dropped connections, slow speeds) are a leading
  voluntary-churn driver, often visible in support interactions before cancellation
- **Payment friction** (manual payment methods) correlates with lower "stickiness"
  than automatic billing

The 10 features below are each grounded in one of these domain realities.

In [ ]:
# 1. Contract commitment level (ordinal domain knowledge)
customers["contract_commitment_level"] = customers["contract"].map(
    {"Month-to-month": 0, "One year": 1, "Two year": 2}
)

# 2. Is the customer in the classic "high churn" segment (no lock-in + manual payment)?
customers["is_high_risk_segment"] = (
    (customers["contract"]=="Month-to-month") &
    (customers["payment_method"].isin(["Electronic check","Mailed check"]))
).astype(int)

# 3. Price-to-service ratio: paying premium (fiber) prices without premium contract commitment
customers["premium_no_lockin"] = (
    (customers["internet_service"]=="Fiber optic") & (customers["contract"]=="Month-to-month")
).astype(int)

# 4. Tenure in months (core recency-independent loyalty measure)
customers["tenure_months_calc"] = ((REF - customers["signup_date"]).dt.days // 30).clip(0, 100)

# 5. New-customer risk window flag (first 90 days = classic high-churn period industry-wide)
customers["in_new_customer_risk_window"] = (customers["tenure_months_calc"] <= 3).astype(int)

# 6. Autopay adoption (proxy for account "stickiness" / reduced friction to leave)
customers["has_autopay"] = customers["payment_method"].str.contains("automatic").astype(int)

# 7. Bundle depth: how many services is the customer using? (more bundled services = higher switching cost)
customers["service_bundle_count"] = (
    (customers["phone_service"]=="Yes").astype(int) +
    (customers["internet_service"]!="No").astype(int) +
    (customers["multiple_lines"]=="Yes").astype(int)
)

# 8. Household context: partner + dependents (higher stability, historically lower voluntary churn)
customers["household_stability_flag"] = (
    (customers["partner"]=="Yes") | (customers["dependents"]=="Yes")
).astype(int)

# 9. Support engagement signal (from Notebook 8's keyword features)
churn_kw = ["cancel","expensive","slow","not working","disconnect"]
customers["support_dissatisfaction_signal"] = customers["support_ticket_text"].str.lower().apply(
    lambda s: sum(kw in s for kw in churn_kw)
)

# 10. Spend trajectory: recent transaction engagement relative to tenure
agg = transactions[transactions["transaction_date"] < REF].groupby("customer_id").agg(
    total_spent=("amount","sum"), tx_count=("amount","count")
).reset_index()
customers = customers.merge(agg, on="customer_id", how="left")
customers[["total_spent","tx_count"]] = customers[["total_spent","tx_count"]].fillna(0)
customers["spend_velocity"] = customers["total_spent"] / customers["tenure_months_calc"].replace(0,1)

feature_summary_cols = ["contract_commitment_level","is_high_risk_segment","premium_no_lockin",
    "tenure_months_calc","in_new_customer_risk_window","has_autopay","service_bundle_count",
    "household_stability_flag","support_dissatisfaction_signal","spend_velocity"]
customers[feature_summary_cols].describe()

## Feature Documentation (Required Format)

| # | Feature Name | Source Column(s) | Logic | Business Meaning | ML Relevance | Leakage Risk | Decision |
|---|---|---|---|---|---|---|---|
| 1 | `contract_commitment_level` | contract | ordinal map 0/1/2 | length of contractual lock-in | Directly captures the single strongest domain-known churn driver | None | **Retain** |
| 2 | `is_high_risk_segment` | contract, payment_method | rule-based AND | combines the two riskiest attributes together | Captures known interaction effect | None | **Retain** |
| 3 | `premium_no_lockin` | internet_service, contract | rule-based AND | fiber customer with no commitment = price-shopping risk | Flags a specific business-known segment | None | **Retain** |
| 4 | `tenure_months_calc` | signup_date | date diff | relationship length | Core loyalty baseline | None (fixed reference date) | **Retain** |
| 5 | `in_new_customer_risk_window` | tenure_months_calc | threshold flag (≤3mo) | onboarding-period churn risk | Industry-known high-churn window | None | **Retain** |
| 6 | `has_autopay` | payment_method | string match | reduced friction to cancel | Proxy for account stickiness | None | **Retain** |
| 7 | `service_bundle_count` | phone_service, internet_service, multiple_lines | count of active services | switching-cost proxy | More bundled = harder to leave | None | **Retain** |
| 8 | `household_stability_flag` | partner, dependents | OR flag | household rootedness | Historically lower voluntary churn | None | **Retain** |
| 9 | `support_dissatisfaction_signal` | support_ticket_text | keyword count | direct dissatisfaction expression | Early warning signal, often precedes cancellation | Low — must ensure ticket predates prediction point | **Retain** |
| 10 | `spend_velocity` | transactions (filtered < REF), tenure_months_calc | ratio | recent engagement rate normalized by tenure | Disengagement (low velocity) precedes churn | None (date-filtered) | **Retain** |

Every feature above passes the domain-thinking test used throughout this sprint: each
one maps to a specific, explainable business hypothesis about *why* a telecom customer
churns — not just a statistically-convenient transformation.